# Import important libraries

In [ ]:
import numpy as np
from pathlib import Path
import torch
import glob
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, Dataloader
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Dataset loading and preprocessing

dataset_root = Path("/kaggle/input/face-mask-dataset/Face Mask Dataset/")

class Kidney_Dataset(Dataset):
	def __init__(self, dataset_root, transform=None):
		self.dataset_root = dataset_root
		self.transform = transform
		self.image_paths = list(self.dataset_root.glob("*/*.jpg"))
		self.labels = [path.parent.name for path in self.image_paths]
		self.label_encoder = self._label_encoder(self.dataset_root)


	def _label_encoder(dataset_root):
		# get labels from the folder names
		class_names = [class_name for class_name in dataset_root.iterdir() if class_name.is_dir()]
		# create a label encoder
		label_encoder = LabelEncoder()
		# build transformation realtionship
		label_encoder.fit(class_names)
		return label_encoder
	
	def encode_label(self, label):
		# label encoder need and output list
		return self.label_encoder.transform([label])[0]
	
	def decode_label(self, encoded_label):
		return self.label_encoder.inverse_transform([encoded_label])[0]
	
	def __len__(self):
		return len(self.image_paths)
	
	def __getitem__(self, idx):
		image_path = self.image_paths[idx]
		label = self.labels[idx]
		
		image = Image.open(image_path).convert("RGB")
		
		if self.transform:
			image = self.transform(image)
		
		encoded_label = self.encode_label(label)
		return image, encoded_label
	


In [ ]:
kidney_dataset = Kidney_Dataset(dataset_root)

# Split the dataset into training and testing sets
train_size = int(0.7 * len(kidney_dataset))
validation_size = int(0.2 * len(kidney_dataset))
test_size = len(kidney_dataset) - train_size - validation_size

# prepare indices for splitting for stratified sampling
dataset_indices = np.arange(len(kidney_dataset))
dataset_labels = [kidney_dataset.labels[i] for i in dataset_indices]

# stratified sampling to maintain class distribution in train, validation and test sets
train_indices, temp_indices = train_test_split(dataset_indices, 
                                               test_size=(validation_size + test_size) / len(kidney_dataset), 
                                               stratify=dataset_labels, 
                                               random_state=42
                                               )

# split validation and test sets
validation_indices, test_indices = train_test_split(temp_indices, 
                                                    test_size = test_size / (validation_size + test_size), 
                                                    stratify=[dataset_labels[i] for i in temp_indices], 
                                                    random_state=42
                                                    )